In [4]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

In [5]:
dados = np.genfromtxt('kc2.csv', delimiter=',')
 
X = dados[:, :-1]
y = dados[:, -1].astype(int)

In [6]:
def distancia_euclidiana(x1, x2):
    return np.sqrt(np.sum((x1 - x2) ** 2))

In [7]:
def distancia_mahalanobis(x1, x2, cov_inv):
    diff = x1 - x2
    return np.sqrt(diff @ cov_inv @ diff)

In [8]:
def calcular_cov_inv(X_train):
    cov = np.cov(X_train, rowvar=False)
    cov += np.eye(cov.shape[0]) * 1e-6
    return np.linalg.inv(cov)

In [9]:
def knn_predict(X_train, y_train, X_test, k, metrica, cov_inv=None):
    y_pred = np.empty(len(X_test), dtype=y_train.dtype)
 
    for i, amostra in enumerate(X_test):
        if metrica == 'euclidean':
            distancias = np.array([distancia_euclidiana(amostra, pt) for pt in X_train])
        else:
            distancias = np.array([distancia_mahalanobis(amostra, pt, cov_inv) for pt in X_train])
 
        k_idx    = np.argsort(distancias)[:k]
        k_labels = y_train[k_idx]
 
        valores, contagens = np.unique(k_labels, return_counts=True)
        y_pred[i] = valores[np.argmax(contagens)]
 
    return y_pred

### Cálculo de métricas

In [10]:
def calcular_metricas(y_real, y_pred, classe_positiva=1):

    VP = np.sum((y_pred == classe_positiva) & (y_real == classe_positiva)) # verdadeiros positivos
    FP = np.sum((y_pred == classe_positiva) & (y_real != classe_positiva)) # falsos positivos
    VN = np.sum((y_pred != classe_positiva) & (y_real != classe_positiva)) # verdadeiros negativos 
    FN = np.sum((y_pred != classe_positiva) & (y_real == classe_positiva)) # falsos negativos
 
    acuracia = (VP + VN) / (VP + FP + VN + FN) if (VP + FP + VN + FN) > 0 else 0.0
    revocacao = VP / (VP + FN) if (VP + FN) > 0 else 0.0
    precisao = VP / (VP + FP) if (VP + FP) > 0  else 0.0
    f1 = (2 * precisao * revocacao / (precisao + revocacao) if (precisao + revocacao) > 0 else 0.0)
 
    return acuracia, revocacao, precisao, f1

### Validação cruzada

In [11]:
def gerar_folds_estratificados(y, n_folds=10, seed=42):

    rng = np.random.default_rng(seed)
    classes = np.unique(y)
 
    indices_por_classe = {}
    for cls in classes:
        idx = np.where(y == cls)[0]
        rng.shuffle(idx)
        indices_por_classe[cls] = idx
 
    folds_por_classe = {cls: np.array_split(idx, n_folds)
                        for cls, idx in indices_por_classe.items()}
 
    folds = []
    for f in range(n_folds):
        idx_teste  = np.concatenate([folds_por_classe[cls][f] for cls in classes])
        idx_treino = np.concatenate([
            np.concatenate([folds_por_classe[cls][j]
                            for j in range(n_folds) if j != f])
            for cls in classes
        ])
        folds.append((idx_treino, idx_teste))
 
    return folds

In [12]:
folds = gerar_folds_estratificados(y, n_folds=10, seed=42)
nomes_metricas = ['Acurácia', 'Revocação', 'Precisão', 'F1-score']
 
def avaliar_arvore(X_train, y_train, X_test, criterio):
    modelo = DecisionTreeClassifier(criterion=criterio, random_state=42)
    modelo.fit(X_train, y_train)
    return modelo.predict(X_test)
 
modelos = [
    ('KNN k=1 Euclidiana',
     lambda Xtr, ytr, Xte: knn_predict(Xtr, ytr, Xte, k=1, metrica='euclidean')),
 
    ('KNN k=5 Euclidiana',
     lambda Xtr, ytr, Xte: knn_predict(Xtr, ytr, Xte, k=5, metrica='euclidean')),
 
    ('KNN k=1 Mahalanobis',
     lambda Xtr, ytr, Xte: knn_predict(Xtr, ytr, Xte, k=1, metrica='mahalanobis', cov_inv=calcular_cov_inv(Xtr))),
 
    ('KNN k=5 Mahalanobis',
     lambda Xtr, ytr, Xte: knn_predict(Xtr, ytr, Xte, k=5, metrica='mahalanobis', cov_inv=calcular_cov_inv(Xtr))),
 
    ('DecisionTree Gini',
     lambda Xtr, ytr, Xte: avaliar_arvore(Xtr, ytr, Xte, 'gini')),
 
    ('DecisionTree Entropy',
     lambda Xtr, ytr, Xte: avaliar_arvore(Xtr, ytr, Xte, 'entropy')),
]
 
resultados = {}
 
for nome, preditor in modelos:
    print(f"Avaliando: {nome}")
    scores = [[] for _ in nomes_metricas]
 
    for idx_treino, idx_teste in folds:
        X_train, X_test = X[idx_treino], X[idx_teste]
        y_train, y_test = y[idx_treino], y[idx_teste]
 
        y_pred = preditor(X_train, y_train, X_test)
 
        for i, val in enumerate(calcular_metricas(y_test, y_pred)):
            scores[i].append(val)
 
    resultados[nome] = {
        metrica: (np.mean(vals), np.std(vals))
        for metrica, vals in zip(nomes_metricas, scores)
    }
 
print()

Avaliando: KNN k=1 Euclidiana
Avaliando: KNN k=5 Euclidiana
Avaliando: KNN k=1 Mahalanobis
Avaliando: KNN k=5 Mahalanobis
Avaliando: DecisionTree Gini
Avaliando: DecisionTree Entropy



In [13]:
header = f"{'Modelo':<26} {'Métrica':<12} {'Média':>8} {'Desvio':>8}"
sep    = "-" * len(header)
 
print(sep)
print(header)
print(sep)
 
for nome, metricas in resultados.items():
    primeiro = True
    for metrica, (media, desvio) in metricas.items():
        rotulo = nome if primeiro else ""
        print(f"{rotulo:<26} {metrica:<12} {media:>8.4f} {desvio:>8.4f}")
        primeiro = False
    print(sep)

---------------------------------------------------------
Modelo                     Métrica         Média   Desvio
---------------------------------------------------------
KNN k=1 Euclidiana         Acurácia       0.6923   0.0799
                           Revocação      0.6464   0.1431
                           Precisão       0.7157   0.0935
                           F1-score       0.6719   0.0990
---------------------------------------------------------
KNN k=5 Euclidiana         Acurácia       0.7486   0.0871
                           Revocação      0.7282   0.0895
                           Precisão       0.7722   0.1188
                           F1-score       0.7447   0.0828
---------------------------------------------------------
KNN k=1 Mahalanobis        Acurácia       0.7409   0.0953
                           Revocação      0.6927   0.1293
                           Precisão       0.7733   0.1104
                           F1-score       0.7250   0.1056
--------------